# Section 1 — CBF Evaluation
Average Top-10 Cosine Similarity across 100 random manga.

In [1]:
import joblib, numpy as np, pandas as pd

cosine_sim = joblib.load('../models/cosine_sim.joblib')
manga_df = pd.read_csv('../models/manga_indexed.csv')

# Average Top-10 Cosine Similarity across 100 random manga
sample_idx = np.random.choice(len(manga_df), 100, replace=False)
avg_sims = []
for idx in sample_idx:
    top10 = np.sort(cosine_sim[idx])[::-1][1:11]
    avg_sims.append(top10.mean())

print(f"CBF Avg Top-10 Cosine Similarity: {np.mean(avg_sims):.4f}")

CBF Avg Top-10 Cosine Similarity: 0.1705


# Section 2 — CF Evaluation
RMSE and MAE for the SVD Collaborative Filtering model.

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

manga_df = pd.read_csv('../models/manga_indexed.csv')
manga_ids = manga_df['id'].tolist()

np.random.seed(42)
rating_weights = [0.02,0.03,0.05,0.07,0.10,0.15,0.20,0.20,0.12,0.06]
rating_values = [1,2,3,4,5,6,7,8,9,10]
records = []
for uid in range(1, 201):
    n = np.random.randint(10, 50)
    rated = np.random.choice(manga_ids, size=min(n, len(manga_ids)), replace=False)
    for mid in rated:
        records.append({'user_id': uid, 'manga_id': int(mid),
                        'rating': np.random.choice(rating_values, p=rating_weights)})
full_df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)

split = int(len(full_df) * 0.8)
train_df, test_df = full_df.iloc[:split], full_df.iloc[split:]

user_map = {u:i for i,u in enumerate(train_df['user_id'].unique())}
manga_map = {m:i for i,m in enumerate(train_df['manga_id'].unique())}
u_idx = train_df['user_id'].map(user_map)
m_idx = train_df['manga_id'].map(manga_map)
mat = csr_matrix((train_df['rating'], (u_idx, m_idx)),
                  shape=(len(user_map), len(manga_map)))

k = min(20, min(mat.shape) - 1)
U, sigma, Vt = svds(mat.astype(float), k=k)
pred_matrix = U @ np.diag(sigma) @ Vt

errors = []
for _, row in test_df.iterrows():
    if row['user_id'] in user_map and row['manga_id'] in manga_map:
        pred = pred_matrix[user_map[row['user_id']], manga_map[row['manga_id']]]
        errors.append((row['rating'], pred))

actuals = np.array([e[0] for e in errors])
preds = np.array([e[1] for e in errors])
rmse = np.sqrt(np.mean((actuals - preds)**2))
mae = np.mean(np.abs(actuals - preds))
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 6.7976
MAE:  6.4483


# Section 3 — Hybrid Precision@10
For each of 50 test users, hide 20% of their high-rated manga, ask hybrid for top-10, check how many hidden ones appear.

In [3]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

# Load models
cosine_sim = joblib.load('../models/cosine_sim.joblib')
manga_df = pd.read_csv('../models/manga_indexed.csv')
manga_ids = manga_df['id'].tolist()

# Generate synthetic users
np.random.seed(42)
rating_weights = [0.02,0.03,0.05,0.07,0.10,0.15,0.20,0.20,0.12,0.06]
rating_values  = [1,2,3,4,5,6,7,8,9,10]

records = []
for uid in range(1, 201):
    n = np.random.randint(10, 50)
    rated = np.random.choice(manga_ids, size=min(n, len(manga_ids)), replace=False)
    for mid in rated:
        records.append({'user_id': uid, 'manga_id': int(mid),
                        'rating': np.random.choice(rating_values, p=rating_weights)})
full_df = pd.DataFrame(records)

def precision_at_k(recommended_ids, relevant_ids, k=10):
    if not recommended_ids or not relevant_ids:
        return 0.0
    return len(set(recommended_ids[:k]) & set(relevant_ids)) / k

title_to_idx = {t.lower(): i for i, t in enumerate(manga_df['title'].astype(str))}

precisions = []
# Test on 50 random users
test_users = full_df['user_id'].unique()[:50]

for uid in test_users:
    user_ratings = full_df[full_df['user_id'] == uid]
    
    # Relevant = manga rated 7 or above
    relevant = user_ratings[user_ratings['rating'] >= 7]['manga_id'].tolist()
    if len(relevant) < 3:
        continue
    
    # Hide 20% of relevant manga — these are ground truth
    hide_n = max(1, len(relevant) // 5)
    hidden = relevant[:hide_n]
    seed_pool = user_ratings[~user_ratings['manga_id'].isin(hidden)]
    
    if seed_pool.empty:
        continue
    
    # Use their highest rated non-hidden manga as seed for CBF
    seed_row = seed_pool.loc[seed_pool['rating'].idxmax()]
    seed_id = seed_row['manga_id']
    seed_row_df = manga_df[manga_df['id'] == seed_id]
    
    if seed_row_df.empty:
        continue
    
    seed_title = seed_row_df['title'].values[0].lower()
    if seed_title not in title_to_idx:
        continue
    
    idx = title_to_idx[seed_title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:11]
    recommended_ids = [int(manga_df.iloc[i]['id']) for i, _ in sim_scores]
    
    p = precision_at_k(recommended_ids, hidden, k=10)
    precisions.append(p)

avg_precision = np.mean(precisions) if precisions else 0
print(f"Hybrid Precision@10: {avg_precision:.4f}")
print(f"Evaluated over {len(precisions)} users")

Hybrid Precision@10: 0.0020
Evaluated over 50 users
